In [1]:
! pip install pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.5


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


# Heart Disease Data Pipeline \n
### This notebook presents a clear, step-by-step data pipeline for the Heart Disease dataset, using modular utility scripts for reproducibility and clarity. Each stage is explained and visualized for effective presentation.

## 1. Data Loading\n
### We begin by loading the raw heart disease data using our custom utility function.

In [5]:
import sys
sys.path.append('../src')
import panda as pd
from data_utils import load_heart_data
import eda_utils
import data_cleaning
import feature_engineering

# Load data
df = load_heart_data('../data/raw/heart.csv')
df.head()

ModuleNotFoundError: No module named 'pandas'

## 2. Exploratory Data Analysis (EDA)

We explore the dataset to understand its structure, spot issues, and gain initial insights.


In [ ]:
# Show basic info
eda_utils.show_basic_info(df)

### Missing Values

Checking for missing data helps us plan cleaning strategies.


In [ ]:
eda_utils.check_missing_values(df)


### Categorical Feature Overview

We examine the distribution of categorical variables.


In [ ]:
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'dataset']
eda_utils.value_counts_categoricals(df, categorical_features)


### Unique Values and Cardinality

Understanding cardinality helps with encoding and feature selection.


In [ ]:
eda_utils.unique_values_cardinality(df)


### Outlier Detection

We use the IQR method to identify potential outliers in numerical features.


In [ ]:
numerical_features = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
eda_utils.detect_outliers_iqr(df, numerical_features)


### Feature Distributions

Visualizing distributions helps spot skewness, outliers, and data issues.


In [ ]:
eda_utils.plot_feature_distributions(df, numerical_features)


### Duplicate Rows

Checking for duplicate records.


In [ ]:
eda_utils.check_duplicates(df)


### Zero and Negative Value Summary

Zero or negative values may indicate data entry issues.


In [ ]:
eda_utils.zero_negative_summary(df, numerical_features)


### Correlation with Target

We examine how features correlate with the target variable.


In [ ]:
eda_utils.correlation_with_target(df, target_col='num')


### Mutual Information

Mutual information helps identify the most informative features.


In [ ]:
eda_utils.mutual_information_with_target(df, numerical_features + categorical_features, target_col='num')


## 3. Data Cleaning

Based on EDA findings, we clean the data: remove duplicates, handle missing values, and address outliers.


In [ ]:
# Remove duplicates
df = data_cleaning.remove_duplicates(df)

# Drop missing values (or use fill_missing)
df = data_cleaning.drop_missing(df)
# df = data_cleaning.fill_missing(df, fill_value=0)  # Alternative

# Remove outliers
df = data_cleaning.remove_outliers_iqr(df, numerical_features)


## 4. Feature Engineering

We prepare the data for modeling by binarizing the target, encoding categoricals, scaling, and creating new features.


In [ ]:
# Binarize target
df = feature_engineering.binarize_target(df, target_col='num', new_col='target')

# Encode categoricals
df = feature_engineering.label_encode_columns(df, categorical_features)
# df = feature_engineering.one_hot_encode_columns(df, categorical_features)  # Alternative

# Scale numerical features
df = feature_engineering.scale_numerical_features(df, numerical_features)

# Create interaction features
feature_pairs = [('age', 'chol'), ('trestbps', 'thalch')]  # Example
df = feature_engineering.create_interaction_features(df, feature_pairs)

# Select top features by mutual information
top_features = feature_engineering.select_top_features_by_mi(
    df, numerical_features + categorical_features, target_col='target', top_n=5
)
print('Top features:', top_features)
